# Load trained models + test data

Everything needed to evaluate the BiLSTM ablation lives here. Run the cells
top to bottom and you get:

- `models` — `{combo_name: model}`, each in eval mode (e.g. `models["pos_dep_shape"]`)
- `test_df` — the held-out test rows (never seen during training)
- `test_loader_for(model)` — a `DataLoader` over the test set with the right
  channels for that model

The split was fixed at training time and saved to `models/split.json`, so every
model is evaluated on the exact same articles. Computing the metrics / ablation
table from here is the evaluation step.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence

MODEL_DIR = Path("models")
DATA_PATH = "Dataset + Feature/sequences.parquet"
COL = {"pos": "pos_ids", "dep": "dep_ids", "shape": "shape_ids"}
PAD_IDX = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Model definition

Same architecture used in training — needed to rebuild each model from its
saved weights.

In [ ]:
class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_sizes, emb_dim, hidden_dim, dropout):
        super().__init__()
        self.channels = list(vocab_sizes.keys())
        self.embeddings = nn.ModuleDict({
            ch: nn.Embedding(size, emb_dim, padding_idx=PAD_IDX)
            for ch, size in vocab_sizes.items()
        })
        self.lstm = nn.LSTM(emb_dim * len(self.channels), hidden_dim,
                            batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 2)

    def forward(self, batch, lengths):
        embs = [self.embeddings[ch](batch[ch]) for ch in self.channels]
        x = torch.cat(embs, dim=-1) if len(embs) > 1 else embs[0]

        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True,
                                      enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)

        mask = (torch.arange(out.size(1), device=out.device)[None, :]
                < lengths.to(out.device)[:, None]).unsqueeze(-1)
        pooled = (out * mask).sum(1) / lengths.to(out.device).clamp(min=1)[:, None]
        return self.fc(self.dropout(pooled))

## Load every model

In [ ]:
def load_one(name):
    ckpt = torch.load(MODEL_DIR / f"bilstm_{name}.pt", map_location=device)
    h = ckpt["hparams"]
    model = BiLSTMTagger(ckpt["vocab_sizes"], h["emb_dim"], h["hidden_dim"], h["dropout"])
    model.load_state_dict(ckpt["state_dict"])
    model.max_len = h["max_len"]
    return model.to(device).eval()

models = {
    p.stem.replace("bilstm_", ""): load_one(p.stem.replace("bilstm_", ""))
    for p in sorted(MODEL_DIR.glob("bilstm_*.pt"))
}

for name, m in models.items():
    n = sum(p.numel() for p in m.parameters())
    print(f"{name:<16} channels={'+'.join(m.channels):<14} params={n:,}")

## Test data from the saved split

In [ ]:
split = json.loads((MODEL_DIR / "split.json").read_text())
df = pd.read_parquet(DATA_PATH)
test_df = df.iloc[split["test"]].reset_index(drop=True)
print(f"test set: {len(test_df)} articles   label balance: {test_df['label'].value_counts().to_dict()}")

In [ ]:
MAX_LEN = next(iter(models.values())).max_len   # same cap used in training


class TestSet(Dataset):
    def __init__(self, frame, channels):
        self.channels = channels
        self.data = {
            ch: [np.array(s[:MAX_LEN], dtype=np.int64) for s in frame[COL[ch]]]
            for ch in channels
        }
        self.labels = frame["label"].to_numpy(dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        seqs = {ch: torch.from_numpy(self.data[ch][i]) for ch in self.channels}
        return seqs, len(seqs[self.channels[0]]), self.labels[i]


def _collate(channels):
    def collate(batch):
        seqs, lengths, labels = zip(*batch)
        padded = {ch: pad_sequence([s[ch] for s in seqs], batch_first=True,
                                   padding_value=PAD_IDX) for ch in channels}
        return padded, torch.tensor(lengths), torch.tensor(labels)
    return collate


def test_loader_for(model, batch_size=64):
    """DataLoader over the test set with exactly the channels this model uses."""
    ds = TestSet(test_df, model.channels)
    return DataLoader(ds, batch_size=batch_size, collate_fn=_collate(model.channels))

## Sanity check

Confirms a model loads and runs on the test data. Replace this with your own
evaluation — collect predictions over `test_loader_for(model)` and compare to
the labels to get accuracy / F1 / confusion matrix per model.

In [ ]:
def predict(model, batch_size=64):
    """Return (y_true, y_pred) over the whole test set for one model."""
    loader = test_loader_for(model, batch_size)
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch, lengths, labels in loader:
            batch = {ch: t.to(device) for ch, t in batch.items()}
            logits = model(batch, lengths.to(device))
            y_pred.extend(logits.argmax(1).cpu().tolist())
            y_true.extend(labels.tolist())
    return np.array(y_true), np.array(y_pred)


# demo on one model
m = models["pos_dep_shape"] if "pos_dep_shape" in models else next(iter(models.values()))
y_true, y_pred = predict(m)
print(f"{'+'.join(m.channels)}: ran on {len(y_true)} test articles, "
      f"accuracy {(y_true == y_pred).mean():.4f}")